# Python Environments — Stable Release Telemetry Dashboard

Runs queries filtered to **stable releases >= 1.24.0** only (even minor version, e.g. 1.24.x, 1.26.x, ...).
Pre-release builds (odd minor versions) are excluded.

**Prerequisites:**
1. `pip install -r requirements.txt`
2. `az login` (Azure CLI authentication)
3. `nbstripout --install` (one-time setup to auto-strip notebook outputs before commits)

In [ ]:
from initialize import initialize
from query_runner import run_kql, run_kql_file, load_kql_sections
from IPython.display import display, Markdown
import pandas as pd

# Show all rows and wrap long column text
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

client = initialize()
sections_09 = load_kql_sections("09-stable-manager-availability.kql")
print(f"Connected to Kusto. Loaded {len(sections_09)} sections from 09-stable-manager-availability.kql.")

---
## 1. MANAGER_REGISTRATION.FAILED — Property Validation (stable >= 1.24.0)

For each manager, shows the breakdown of **error types** from registration failures.
`MachinePct` = % of all stable machines that hit this manager + error type combination.

| Error type | Meaning |
|---|---|
| `spawn_timeout` | Tool process hung or didn't respond in time |
| `spawn_enoent` | OS couldn't find the executable (not installed or not on PATH) |
| `permission_denied` | Exists but no permission to run (EACCES/EPERM) |
| `canceled` | Operation stopped (user closed VS Code / CancellationToken fired) |
| `parse_error` | Tool ran but returned unparseable output |
| `process_crash` | Process started but exited with a non-zero code unexpectedly |
| `unknown` | Catch-all for unexpected errors |

In [ ]:
title, query = sections_09[0]
display(Markdown(f"### {title}"))
df_prop = run_kql(client, query)
display(df_prop)

---
## 2. MANAGER_REGISTRATION.FAILED — Total Failures by Manager × Version (stable >= 1.24.0)

Total failure count per manager per extension version, regardless of error type.
`MachinePct` = % of machines on that version that had any registration failure for this manager.
Same manager name appears in consecutive rows, sorted by version ascending.

In [ ]:
title, query = sections_09[1]
display(Markdown(f"### {title}"))
df_failures_by_version = run_kql(client, query)
display(df_failures_by_version)

---
## 3. SPAWN_TIMEOUT Failures by Manager × Version (stable >= 1.24.0)

Shows whether `spawn_timeout` is improving or worsening across stable releases.
If a new stable version shows higher `MachinePct` → that release regressed timeout handling.

In [ ]:
title, query = sections_09[2]
display(Markdown(f"### {title}"))
df_timeout = run_kql(client, query)
display(df_timeout)

---
## 4. Process Crash Failures by Manager × Version (stable >= 1.24.0)

Shows `process_crash` failures across all stable versions.
Versions with no crashes show `EventCount = 0` — confirming the absence of regressions, not missing data.
If a new stable version shows higher `MachinePct` → that release introduced a crash regression.

In [ ]:
title, query = sections_09[3]
display(Markdown(f"### {title}"))
df_crashes = run_kql(client, query)
display(df_crashes)

---
## 5. UNKNOWN Failures by Manager × Version (stable >= 1.24.0)

Shows whether unclassified (`unknown`) errors are improving or worsening across stable releases.
High counts in the latest stable version → new unclassified error paths need investigation and classification.

In [ ]:
title, query = sections_09[4]
display(Markdown(f"### {title}"))
df_unknown = run_kql(client, query)
display(df_unknown)